# Naive Bayes dự đoán hành vi mua hàng

Dữ liệu Customer Behaviour gồm giới tính, tuổi, mức lương ước tính và nhãn `Purchased`. Dùng Gaussian Naive Bayes cho các thuộc tính số sau khi mã hóa giới tính.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, ConfusionMatrixDisplay)

DATA_PATH = Path("../data/Customer_Behaviour.csv")
df = pd.read_csv(DATA_PATH)
print(f"Kích thước dữ liệu: {df.shape}")
display(df.head())
print("\nKiểu dữ liệu và giá trị thiếu:")
display(pd.DataFrame({"kiểu dữ liệu": df.dtypes.astype(str), "giá trị thiếu": df.isna().sum()}))


## Khám phá dữ liệu và chia tập

In [ ]:
TARGET = "Purchased"
print("Phân bố nhãn:")
display(df[TARGET].value_counts(dropna=False).to_frame("số lượng"))
print("\nThống kê thuộc tính số:")
display(df.select_dtypes(include="number").describe().T)
df_model = df.dropna(subset=[TARGET]).copy()
X = df_model.drop(columns=[TARGET]).drop(columns=["User ID"], errors="ignore")
if "Gender" in X.columns:
    X["Gender"] = X["Gender"].map({"Female": 0, "Male": 1})
y = df_model[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)


## Huấn luyện Gaussian Naive Bayes

GaussianNB mô hình hóa phân bố của các thuộc tính số trong từng lớp. Bước điền giá trị thiếu được đặt trong pipeline.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
model = make_pipeline(SimpleImputer(strategy="median"), GaussianNB())
model.fit(X_train, y_train)


## Đánh giá mô hình

Dữ liệu kiểm tra được giữ riêng khỏi quá trình huấn luyện. Báo cáo precision, recall, F1 và ma trận nhầm lẫn cho từng lớp.

In [ ]:
y_pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues", values_format="d")
plt.title("Ma trận nhầm lẫn - Gaussian Naive Bayes")
plt.tight_layout()
plt.show()


## Nhận xét

Dựa trên classification report và ma trận nhầm lẫn ở trên, hãy nhận xét mô hình phân loại **Purchased** tốt đến đâu, đặc biệt chú ý lớp ít xuất hiện và các trường hợp dự đoán sai.